# 🗳️ Voting Schema Experiments — S6E4

**Goal:** Beat the current binary schema ("unanimous or fallback") by exploiting partial agreement signal.

## The Problem
Current schema: if 4 LGB clones agree → use consensus; else → use fallback (7971.x).

This **throws away** rows where 3/4 agree — hundreds of rows of signal!

From E.o.S. output:
- `L _ L L L` → 110 rows (4/5 agree on Low, including fallback)
- `L L L L _` → 7 rows (4/5 agree on Low)
- `L _ L _ L` → 10 rows (3/5 agree on Low)
- `_ L L L L` → 4 rows (4/5 agree on Low)
- `_ _ _ H _` → 76 rows (4/5 agree on Medium)
- And dozens more patterns...

## Schemas to Test

| Schema | Logic | Rationale |
|--------|-------|----------|
| **A. Current (baseline)** | If 4 clones agree → consensus; else → fallback | Proven: 0.97972 |
| **B. Strict majority** | If ≥3 clones agree → majority; else → fallback | Captures 3/4 agreement |
| **C. Cascading** | LGB1+LGB2 agree → trust; else add LGB3; else add LGB4; else fallback | Progressive confidence |
| **D. Weighted** | Each model gets weight; prediction = argmax of weighted votes | Stronger models matter more |
| **E. Pattern-specific** | Custom rules for each mEDA pattern | Exploit empirical patterns |
| **F. Pairwise agreement** | Count agreeing pairs; if ≥3 pairs agree → consensus | More granular than simple majority |

## Methodology
1. Run each schema on the SAME input submissions
2. Save each schema's output as a separate submission file
3. Compare pairwise disagreements between schemas
4. Build a meta-ensemble of the best schemas

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import os

# ─── Paths ─────────────────────────────────────────────────────
PPS = '/kaggle/input/competitions/playground-series-s6e4/'

def find_path(candidates):
    for p in candidates:
        full = p if p.endswith('/') else p + '/'
        try:
            if os.listdir(full): return full
        except: pass
    return candidates[-1] if candidates[-1].endswith('/') else candidates[-1] + '/'

PD7  = find_path(['/kaggle/input/datasets/gajananbarve/ps-s6e4-07/', '/kaggle/input/datasets/nina2025/ps-s6e4-07/'])
PD74 = find_path(['/kaggle/input/datasets/gajananbarve/ps-s6e4-74/', '/kaggle/input/datasets/nina2025/ps-s6e4-74/'])
PD85 = find_path(['/kaggle/input/datasets/gajananbarve/ps-s6e4-85/', '/kaggle/input/datasets/nina2025/ps-s6e4-85/'])

print(f"PD7  = {PD7}")
print(f"PD74 = {PD74}")
print(f"PD85 = {PD85}")

In [ ]:
# ─── Load all submissions ─────────────────────────────────────
df_a   = pd.read_csv(PD7  + '0.97971.a.csv')['Irrigation_Need'].values
df_b   = pd.read_csv(PD7  + '0.97971.b.csv')['Irrigation_Need'].values
df_c   = pd.read_csv(PD7  + '0.97971.c.csv')['Irrigation_Need'].values
df_d   = pd.read_csv(PD7  + '0.97971.d.csv')['Irrigation_Need'].values
df_x   = pd.read_csv(PD7  + '0.97971.x.csv')['Irrigation_Need'].values    # fallback
df_8010 = pd.read_csv(PD7  + '0.98010.csv')['Irrigation_Need'].values
df_8057 = pd.read_csv(PD74 + '5(8) - 0.98057.csv')['Irrigation_Need'].values
df_8074 = pd.read_csv(PD74 + '5(4) - 0.98074.csv')['Irrigation_Need'].values
df_8072 = pd.read_csv(PD85 + '5(9) - 0.98072.csv')['Irrigation_Need'].values
df_aux1 = pd.read_csv(PD85 + 'Aux - 0.97254.csv')['Irrigation_Need'].values

# Stack into a 2D array: rows = test samples, cols = models
models = np.column_stack([df_a, df_b, df_c, df_d, df_x, df_8010, df_8057, df_8074, df_8072, df_aux1])
model_names = ['7971a', '7971b', '7971c', '7971d', '7971x(fallback)', '8010', '8057', '8074', '8072', 'Aux1']
n_models = len(model_names)
n_samples = len(df_a)

print(f"Loaded {n_samples} samples × {n_models} models")
print(f"Models: {model_names}")

## Analyze: Where does the current schema waste signal?

In [ ]:
# ─── Analyze 4-clone agreement patterns ───────────────────────
clones = models[:, :4]  # 7971a, b, c, d
fallback = models[:, 4]  # 7971x

def get_pattern(row):
    return ' '.join([v[0] if v[0] != 'M' else '_' for v in row]) + ' '

# Build patterns for 4 clones + fallback
patterns_5 = []
for i in range(n_samples):
    p = ' '.join([clones[i, j][0] if clones[i, j][0] != 'M' else '_' for j in range(4)])
    p += ' ' + (fallback[i][0] if fallback[i][0] != 'M' else '_') + ' '
    patterns_5.append(p)

pattern_counts = Counter(patterns_5)

# Show patterns where clones DON'T unanimously agree (current schema falls through)
print("=== Patterns where clones DON'T unanimously agree (signal wasted by current schema) ===")
wasted = 0
for pattern, count in sorted(pattern_counts.items(), key=lambda x: -x[1]):
    parts = pattern.strip().split(' ')
    clone_votes = Counter(parts[:4])
    max_agree = clone_votes.most_common(1)[0][1]
    if max_agree < 4:  # Not unanimous
        print(f"  {pattern.strip():15s} → {count:6d} rows  (max clone agreement: {max_agree}/4, fallback={parts[4]})")
        wasted += count

print(f"\nTotal rows where current schema falls through: {wasted:,} / {n_samples:,}")
print(f"Of those, how many have 3/4 clone agreement:")
three_agree = 0
for pattern, count in sorted(pattern_counts.items(), key=lambda x: -x[1]):
    parts = pattern.strip().split(' ')
    clone_votes = Counter(parts[:4])
    max_agree = clone_votes.most_common(1)[0][1]
    if max_agree == 3:
        print(f"  {pattern.strip():15s} → {count:6d} rows  (3/4 agree on {clone_votes.most_common(1)[0][0]}, fallback={parts[4]})")
        three_agree += count
print(f"  Total with 3/4 agreement: {three_agree:,}")

## Define all voting schemas

In [ ]:
# ─── Schema A: Current (baseline) ─────────────────────────────
def schema_A(clones, fallback):
    """If all 4 clones agree → consensus; else → fallback."""
    preds = []
    for i in range(n_samples):
        c = [clones[i, j] for j in range(4)]
        if c[0] == c[1] == c[2] == c[3]:
            preds.append(c[0])
        else:
            preds.append(fallback[i])
    return preds

# ─── Schema B: Strict majority (≥3/4 clones agree → trust majority) ─
def schema_B(clones, fallback):
    """If ≥3 clones agree → majority; else → fallback."""
    preds = []
    for i in range(n_samples):
        c = [clones[i, j] for j in range(4)]
        counter = Counter(c)
        top = counter.most_common(1)[0]
        if top[1] >= 3:
            preds.append(top[0])
        else:
            preds.append(fallback[i])
    return preds

# ─── Schema C: Cascading (progressive confidence) ─────────────
def schema_C(clones, fallback):
    """Add clones one by one until majority emerges; else fallback."""
    preds = []
    for i in range(n_samples):
        # Try 1+2
        if clones[i, 0] == clones[i, 1]:
            preds.append(clones[i, 0])
            continue
        # Add 3
        c3 = [clones[i, 0], clones[i, 1], clones[i, 2]]
        counter3 = Counter(c3)
        if counter3.most_common(1)[0][1] >= 2:
            preds.append(counter3.most_common(1)[0][0])
            continue
        # Add 4
        c4 = c3 + [clones[i, 3]]
        counter4 = Counter(c4)
        if counter4.most_common(1)[0][1] >= 2:
            preds.append(counter4.most_common(1)[0][0])
            continue
        # All different → fallback
        preds.append(fallback[i])
    return preds

# ─── Schema D: Weighted voting ────────────────────────────────
def schema_D(clones, fallback, weights=None):
    """Each clone gets a weight; prediction = argmax of weighted votes."""
    if weights is None:
        weights = [1.0, 1.0, 1.0, 1.0]  # Equal weights for clones
    preds = []
    for i in range(n_samples):
        weighted = defaultdict(float)
        for j in range(4):
            weighted[clones[i, j]] += weights[j]
        # Also add fallback with lower weight
        weighted[fallback[i]] += 0.5
        preds.append(max(weighted, key=weighted.get))
    return preds

# ─── Schema E: Pattern-specific rules ─────────────────────────
def schema_E(clones, fallback):
    """Custom rules for each mEDA pattern."""
    preds = []
    for i in range(n_samples):
        c = [clones[i, j] for j in range(4)]
        fb = fallback[i]
        counter = Counter(c)
        top = counter.most_common(1)[0]
        second = counter.most_common(2)[1] if len(counter) > 1 else None
        
        if top[1] == 4:
            # Unanimous → trust
            preds.append(top[0])
        elif top[1] == 3:
            if top[0] == fb:
                # Majority agrees with fallback → very confident
                preds.append(top[0])
            elif top[0] in ['Low', 'High']:
                # Extreme majority → trust majority over fallback
                preds.append(top[0])
            else:
                # Medium majority → trust fallback
                preds.append(fb)
        elif top[1] == 2 and second and second[1] == 2:
            # 2 vs 2 split → trust fallback
            preds.append(fb)
        elif top[1] == 2:
            # 2 vs 1 vs 1 → check if majority matches fallback
            if top[0] == fb:
                preds.append(top[0])
            else:
                preds.append(fb)
        else:
            # All different → fallback
            preds.append(fb)
    return preds

# ─── Schema F: Pairwise agreement ─────────────────────────────
def schema_F(clones, fallback):
    """Count agreeing pairs; high pairwise agreement → consensus."""
    preds = []
    for i in range(n_samples):
        c = [clones[i, j] for j in range(4)]
        # Count pairwise agreements
        agreements = 0
        total_pairs = 0
        for a in range(4):
            for b in range(a+1, 4):
                total_pairs += 1
                if c[a] == c[b]:
                    agreements += 1
        
        counter = Counter(c)
        top = counter.most_common(1)[0]
        
        if agreements >= 5:  # ≥5/6 pairs agree → strong consensus
            preds.append(top[0])
        elif agreements >= 3:  # Moderate agreement → check fallback
            if top[0] == fallback[i] or top[1] >= 3:
                preds.append(top[0])
            else:
                preds.append(fallback[i])
        else:
            preds.append(fallback[i])
    return preds

print("All schemas defined.")

## Run all schemas

In [ ]:
# ─── Run all schemas ──────────────────────────────────────────
schemas = {
    'A_current': schema_A(clones, fallback),
    'B_strict_majority': schema_B(clones, fallback),
    'C_cascading': schema_C(clones, fallback),
    'D_weighted': schema_D(clones, fallback),
    'E_pattern_specific': schema_E(clones, fallback),
    'F_pairwise': schema_F(clones, fallback),
}

# Print distribution for each
print(f"{'Schema':<25s} {'Low':>8s} {'Medium':>8s} {'High':>8s}  Changed vs A")
print("-" * 75)
baseline = schemas['A_current']
for name, preds in schemas.items():
    counter = Counter(preds)
    n_changed = sum(1 for a, b in zip(baseline, preds) if a != b)
    print(f"{name:<25s} {counter.get('Low',0):8d} {counter.get('Medium',0):8d} {counter.get('High',0):8d}  {n_changed:6d}")

## Cross-schema disagreement analysis

Where schemas disagree is where the "truth" likely lives.

In [ ]:
# ─── Cross-schema disagreement matrix ─────────────────────────
schema_names = list(schemas.keys())
n_schemas = len(schema_names)

print("=== Pairwise Disagreement Matrix ===")
print(f"{'':>20s}", end='')
for name in schema_names:
    print(f" {name[:8]:>8s}", end='')
print()

for i, name_a in enumerate(schema_names):
    print(f"{name_a:>20s}", end='')
    for j, name_b in enumerate(schema_names):
        if i == j:
            print(f" {'—':>8s}", end='')
        else:
            n_diff = sum(1 for a, b in zip(schemas[name_a], schemas[name_b]) if a != b)
            print(f" {n_diff:>8d}", end='')
    print()

In [ ]:
# ─── Find rows where ALL schemas agree (high confidence) ──────
all_agree = np.ones(n_samples, dtype=bool)
for name, preds in schemas.items():
    all_agree &= (np.array(preds) == np.array(schemas['A_current']))

print(f"Rows where ALL schemas agree: {all_agree.sum():,} / {n_samples:,}")
print(f"Rows where at least one schema differs: {(~all_agree).sum():,}")

# ─── Find rows where MOST schemas agree but one differs ────────
pred_matrix = np.array([[schemas[n][i] for n in schema_names] for i in range(n_samples)])

disagreement_rows = []
for i in range(n_samples):
    row_preds = pred_matrix[i]
    counter = Counter(row_preds)
    top = counter.most_common(1)[0]
    if top[1] >= 5:  # At least 5 out of 6 agree
        # Find the dissenter
        dissenters = [schema_names[j] for j in range(n_schemas) if row_preds[j] != top[0]]
        if dissenters:
            disagreement_rows.append({
                'row_idx': i,
                'majority': top[0],
                'majority_count': top[1],
                'dissenters': ', '.join(dissenters),
                'dissenting_value': row_preds[[schema_names.index(d) for d in dissenters][0]]
            })

df_disagree = pd.DataFrame(disagreement_rows)
if len(df_disagree) > 0:
    print(f"\nRows with 5/6 agreement (dissenters can be corrected):")
    print(f"  Total: {len(df_disagree):,}")
    print(f"  Dissenters breakdown:")
    print(df_disagree['dissenters'].value_counts().head(15).to_string())
    print(f"\n  What the dissenters predicted:")
    print(df_disagree['dissenting_value'].value_counts().to_string())

## Meta-ensemble: Combine the best schemas

In [ ]:
# ─── Meta-ensemble: For each row, take majority vote across schemas ─
def meta_ensemble(schemas_dict):
    preds = []
    for i in range(n_samples):
        votes = [schemas_dict[n][i] for n in schemas_dict]
        counter = Counter(votes)
        preds.append(counter.most_common(1)[0][0])
    return preds

meta_all = meta_ensemble(schemas)
meta_no_A = meta_ensemble({k: v for k, v in schemas.items() if k != 'A_current'})

print("=== Meta-ensemble Results ===")
print(f"{'Schema':<25s} {'Low':>8s} {'Medium':>8s} {'High':>8s}  Diff vs A")
print("-" * 75)

for name, preds in [('Meta_all_6', meta_all), ('Meta_no_A_5', meta_no_A)]:
    counter = Counter(preds)
    n_changed = sum(1 for a, b in zip(schemas['A_current'], preds) if a != b)
    print(f"{name:<25s} {counter.get('Low',0):8d} {counter.get('Medium',0):8d} {counter.get('High',0):8d}  {n_changed:6d}")

In [ ]:
# ─── Generate submission files for each schema ────────────────
sub_template = pd.read_csv(PPS + 'sample_submission.csv')

os.makedirs('schema_outputs', exist_ok=True)

for name, preds in schemas.items():
    sub = sub_template.copy()
    sub['Irrigation_Need'] = preds
    fname = f'schema_outputs/submission_{name}.csv'
    sub.to_csv(fname, index=False)
    print(f"  Saved {fname}")

# Meta-ensembles
for name, preds in [('meta_all_6', meta_all), ('meta_no_A_5', meta_no_A)]:
    sub = sub_template.copy()
    sub['Irrigation_Need'] = preds
    fname = f'schema_outputs/submission_{name}.csv'
    sub.to_csv(fname, index=False)
    print(f"  Saved {fname}")

print(f"\n✅ Generated {len(schemas) + 2} submission files in schema_outputs/")
print("Upload each to Kaggle to find the LB winner!")

## Bonus: Combine with barrel + aux correction from before

Take the best schema + barrel split + auxiliary correction for the full pipeline.

In [ ]:
# ─── Full optimized pipeline with best schema ─────────────────
# Use Schema E (pattern-specific) as it exploits the most signal
best_schema_name = 'E_pattern_specific'
best_preds = schemas[best_schema_name]

print(f"Using {best_schema_name} as base for full pipeline")
print(f"Distribution: {Counter(best_preds)}")

# Now apply barrel split optimization
df74_preds = df_8074
df72_preds = df_8072
base_preds = np.array(best_preds)

print("\nScanning barrel split points...")
best_split = 137400
best_score = float('inf')

for split in range(120000, 155000, 1000):
    blended = np.concatenate([df74_preds[:split], base_preds[split:]])
    d74 = np.sum(blended != df74_preds)
    d72 = np.sum(blended != df72_preds)
    if d74 + d72 < best_score:
        best_score = d74 + d72
        best_split = split

print(f"  Best split: {best_split:,}")

# Apply barrel split
barrel_preds = np.concatenate([df74_preds[:best_split], base_preds[best_split:]])

# Apply auxiliary correction on df74 vs df72 disagreement rows
disagree_mask = df_8074 != df_8072
n_disagree = disagree_mask.sum()
print(f"  Disagreement rows: {n_disagree:,}")

# 3-way auxiliary vote on disagreement rows
for i in range(n_samples):
    if disagree_mask[i]:
        votes = [df_aux1[i], barrel_preds[i]]
        # If Aux1 and barrel agree → use it
        if votes[0] == votes[1]:
            barrel_preds[i] = votes[0]
        # If they disagree → keep barrel (stronger)

# Save final submission
sub = sub_template.copy()
sub['Irrigation_Need'] = barrel_preds
sub.to_csv('submission.csv', index=False)
print(f"\n✅ Final submission.csv saved!")
print(f"   Schema: {best_schema_name}")
print(f"   Barrel split: {best_split:,}")
print(f"   Distribution: {Counter(barrel_preds)}")